In the preivous lab session, we have learned how to build a general understanding of Differential Privacy in the context of a database query. We have learned whether a database query over such a small database is differentially private or not and implemented a local differential privacy to defend against differential attacks. **In this lab session, we introduce the concept of global (centralized) differential privacy that adds noise to the qeury result rather than adding noise to the individual data points.** Have a try:)

**Objectives**

1. Create a simple database and generate parallel databases
2. Create qeury functions and calculate sensitivity
3. Attempt to complete serveral differential attacks (Review Lab 3)
4. Implement **global (centralized) differential privacy**

# Preparation
## Step 1: create a simple database


In [1]:
import numpy as np

# the number of entries in our database
num_entries = 1000

db = np.random.random_sample(num_entries) > 0.5
db

array([ True,  True,  True,  True,  True,  True,  True,  True, False,
        True, False,  True, False,  True,  True, False, False,  True,
       False, False,  True,  True, False,  True, False, False,  True,
       False, False,  True, False, False, False,  True,  True, False,
       False, False, False,  True, False,  True, False,  True,  True,
       False,  True,  True,  True,  True, False, False,  True, False,
       False, False, False,  True, False, False,  True,  True,  True,
        True, False,  True, False,  True, False, False,  True,  True,
       False, False,  True,  True, False,  True, False, False,  True,
        True, False,  True,  True, False, False,  True, False, False,
       False,  True,  True, False, False, False,  True, False,  True,
        True,  True,  True,  True, False,  True,  True,  True, False,
       False,  True, False, False, False, False,  True, False, False,
       False, False, False, False,  True,  True, False, False,  True,
       False,  True,

## Step 2: generate parrallel databases

In [2]:
def get_parrallel_db(db, remove_index):
    pdb = list(db)
    # remove the entry by index
    del pdb[remove_index]
    return np.array(pdb)

In [3]:
get_parrallel_db(db, 2) # generate a parralleb database by removing the 3rd entry in the intial database
print('Size of intial db:',len(db))
print('Size of parrallel db:',len(get_parrallel_db(db, 2)))

Size of intial db: 1000
Size of parrallel db: 999


In [4]:
def get_parrallel_dbs(db):
    parrallel_dbs = list()
    for i in range(len(db)):
        # each parrallel database removes the ith entry from the intial database
        pdb = get_parrallel_db(db,i)
        parrallel_dbs.append(pdb)
    return parrallel_dbs

In [5]:
def create_db_and_parrallels(num_entries):
    db = np.random.random_sample(num_entries) > 0.5
    pdbs = get_parrallel_dbs(db)
    return db, pdbs

## Step 3: calculate the sensitivity in terms of query functions

In [6]:
def query_sum(db):
    return db.sum()

In [7]:
def sensitivity_sum(n_entries):
    # generate the initial database and all the possible parrallel databases
    db, pdbs = create_db_and_parrallels(n_entries)

    # sum value of the intial database
    full_db_result = query_sum(db)

    maximum_distance = 0
    for pdb in pdbs:
        # sum value of each parrallel database
        pdb_result = query_sum(pdb)

        # the difference between the sum values of the initial and each parrallel database
        db_distance = np.abs(pdb_result-full_db_result)
        # find out and return the maximum difference from all those differences
        if(db_distance > maximum_distance):
            maximum_distance = db_distance
    return maximum_distance

In [8]:
sensitivity_sum(1000)

np.int64(1)

In [9]:
# calculate the mean value of the entries in a dataset
def query_mean(db):
    return db.mean()

In [10]:
def sensitivity_mean(n_entries):
    # generate the initial database and all the possible parrallel databases
    db, pdbs = create_db_and_parrallels(n_entries)

    # mean value of the intial database
    full_db_result = query_mean(db)

    maximum_distance = 0
    for pdb in pdbs:
        # mean value of each parrallel database
        pdb_result = query_mean(pdb)

        # the difference between the mean values of the initial and each parrallel database
        db_distance = np.abs(pdb_result-full_db_result)

        # find out and return the maximum difference from all those differences
        if(db_distance > maximum_distance):
            maximum_distance = db_distance
    return maximum_distance

In [11]:
sensitivity_mean(1000)

np.float64(0.0005025025025024998)

In [12]:
def query_threshold(db, threshold=5):
    if db.sum()>threshold:
        return 1
    else:
        return 0

In [13]:
def sensitivity_threshold(n_entries):
    # generate the initial database and all the possible parrallel databases
    db, pdbs = create_db_and_parrallels(n_entries)

    # boolean value if the sum value of the initial database is greater than the threshold
    full_db_result = query_threshold(db)

    maximum_distance = 0
    for pdb in pdbs:
        # boolean value if the sum value of each parrallel database is greater than the threshold
        pdb_result = query_threshold(pdb)

        # the difference between the boolean values of the initial and each parrallel database
        db_distance = np.abs(pdb_result-full_db_result)

        # find out and return the maximum difference from all those differences
        if(db_distance > maximum_distance):
            maximum_distance = db_distance
    return maximum_distance

In [14]:
for i in range(10):
    print(sensitivity_threshold(10))

0
0
0
0
0
0
0
1
0
0


## Step 4: define a differential attack

In [15]:
db,_ = create_db_and_parrallels(100)

For example, let us perform a differential attack onn ***Row 10***. So let us create a prrallel database where just the ***Row 10*** is missing.

In [16]:
pdb = get_parrallel_db(db, remove_index=10)

Let us check what is the value of ***Row 10*** in the initial database.

In [17]:
db[10]

np.False_

In [18]:
# differential attack using sum query
sum(db) - sum(pdb)

np.int64(0)

From the sum query attack, we can know that if Row 10 is True (1), then the sum value is 1, otherwise is 0.

In [19]:
# differential attack using mean query
sum(db)/len(db) -sum(pdb)/len(pdb)

np.float64(-0.005252525252525286)

From the mean query attack, we can know that if Row 10 is True (1), then the mean value is none-zero, otherwise is 0.

In [20]:
# differential attack using threshold
threshold = sum(db) - 1
sum(pdb) > threshold

np.True_

## Step 5: try global (centralized) differential privacy (major)

So the idea of global differential privacy is to add noise to the qeury result rather than adding noise to the individual data points. In other words, it gives a noise to the query result of the entire database. The formal definition of differential noise is as below:



Let M be a randomized algorithm that takes a dataset D as input and outputs a result in some output space O. The algorithm M satisfies (ε, δ)-differential privacy if, for all parallel (neighboring) datasets D and D' (i.e., datasets that differ in only one element), and for all measurable subsets S ⊆ O:

Pr[M(D) ∈ S] ≤ exp(ε) * Pr[M(D') ∈ S] + δ

Where:

- **ε (epsilon)** is a small positive parameter that controls the privacy loss. Smaller ε implies stronger privacy.
- **δ (delta)** is another small parameter that accounts for a small probability of failure (i.e., when differential privacy might not hold).
- **D and D'** are neighboring datasets, differing by at most one entry (typically a single individual’s data).
- **Pr[M(D) ∈ S]** is the probability that the algorithm M outputs a result within the set S, given input dataset D.

### Explanation:
- **Epsilon (ε)** quantifies the worst-case privacy loss. A smaller value of ε provides stronger privacy guarantees but typically comes with less accurate results.
- **Delta (δ)** provides a probabilistic relaxation to the guarantee. It allows for a small chance that the privacy guarantee is broken, but this probability is very small.

In simple terms, differential privacy ensures that the output of an algorithm is almost indistinguishable whether or not any individual’s data is included in the dataset, thus preserving privacy.


### Epsilon $\varepsilon$

Let's unpack the intuition of this for a moment.

Epsilon Zero: If a query satisfied this inequality where epsilon was set to 0, then that would mean that the query for all parallel databases outputed the exact same value as the full database. As you may remember, when we calculated the "threshold" function, often the Sensitivity was 0. In that case, the epsilon also happened to be zero.

Epsilon One: If a query satisfied this inequality with epsilon 1, then the maximum distance between all queries would be 1 - or more precisely - the maximum distance between the two random distributions M(x) and M(y) is 1 (because all these queries have some amount of randomness in them, just like we observed in the last section).

### Delta $\delta$

Delta is basically the probability that epsilon breaks. Namely, sometimes the epsilon is different for some queries than it is for others. For example, you may remember when we were calculating the sensitivity of threshold, most of the time sensitivity was 0 but sometimes it was 1. Thus, we could calculate this as "epsilon zero but non-zero delta" which would say that epsilon is perfect except for some probability of the time when it's arbitrarily higher. Note that this expression doesn't represent the full tradeoff between epsilon and delta.

Now, we're going to learn about how to take a query and add varying amounts of noise so that it satisfies a certain degree of differential privacy. In particular, we're going to leave behind the Local Differential privacy previously discussed and instead opt to focus on Global differential privacy.

So, to sum up, this lesson is about adding noise to the output of our query so that it satisfies a certain epsilon-delta differential privacy threshold.

There are two kinds of noise we can add - Gaussian Noise or Laplacian Noise. Generally speaking Laplacian is better, but both are still valid. Now to the hard question...

### How much noise should we add?

The amount of noise necessary to add to the output of a query is a function of four things:

1. the type of noise (Exponential/Laplacian)
2. the sensitivity of the query/function
3. the desired epsilon (ε)
4. the desired delta (δ)

Thus, for each type of noise we're adding, we have different way of calculating how much to add as a function of sensitivity, epsilon, and delta. We're going to focus on Laplacian noise. Laplacian noise is increased/decreased according to a "scale" parameter b. We choose "b" based on the following formula.

b = sensitivity(query) / epsilon

In other words, if we set b to be this value, then we know that we will have a privacy leakage of <= epsilon. Furthermore, the nice thing about Laplace is that it guarantees this with delta == 0. There are some tunings where we can have very low epsilon where delta is non-zero, but we'll ignore them for now.

Now let us perform a global differential privacy mechanism. First, we create a database with 100 entries and its 99 parrallel database with 99 entries.

In [21]:
entries_num = 100
db,pdbs = create_db_and_parrallels(entries_num)

Then we define a more generalised function to calculate the sensitivity for all query functions.

In [22]:
def sensitivity(dp, pdbs, query):
    # query value of the intial database
    full_db_result = query(db)

    maximum_distance = 0
    for pdb in pdbs:
        # query value of each parrallel database
        pdb_result = query(pdb)

        # the difference between the query values of the initial and each parrallel database
        db_distance = np.abs(pdb_result-full_db_result)

        # find out and return the maximum difference from all those differences
        if(db_distance > maximum_distance):
            maximum_distance = db_distance
    return maximum_distance

Third, let us assign a value to epsilon

In [23]:
epsilon = 0.5

Fourth, write down the laplacian mechanism according to its definition: $$\beta=\frac{sensitivity}{\varepsilon}$$

Moreover, you can use numpy.laplace() to draw samples from laplacian distribution.

In [24]:
def laplacian_mechanism(db, query, sensitivity):
    beta = sensitivity/epsilon
    noise = np.random.laplace(0,beta,1)

    return query(db)+noise

In [25]:
# true query result
print('True Sum:', query_sum(db))
print('True Mean:', query_mean(db))

True Sum: 52
True Mean: 0.52


In [26]:
# query result with laplacian noise
print('Noised Sum:', laplacian_mechanism(db, query_sum, sensitivity(db, pdbs, query_sum)))
print('Noised Mean:', laplacian_mechanism(db, query_mean, sensitivity(db, pdbs, query_mean)))

Noised Sum: [56.24481098]
Noised Mean: [0.53260193]


As you can see, even though we only have 100 entries, the deviation from the true query result is still very small compared to local differential privacy. This is because that global differential privacy only adds noise to the query result which add far less noise than local differential privacy mechanism.

Then let us try another global differential privacy mechanism - exponential noise. Exponential mechanism is used for non-numerical query. For example, we can use Laplacian mechanism for the query 'how many people's answer is yes?'. But for the query such as 'what is the favourite answer ({yes, no}) for this dataset?', we should use exponential mechanism.

The definition of expoenntial_mechanism is $Pr = \frac{\varepsilon*Q(D,r)}{2\Delta Q}$ where D is the dataset, and $r$ is the output for $D$. $Q(D,r)$ can be seen as how good the output $r$ is for $D$.

Again, we create a database with 100 entries and its 99 parrallel database with 99 entries.

In [27]:
entries_num = 100
db,pdbs = create_db_and_parrallels(entries_num)

Then, we can calculate how many answers are 'yes'(True) and 'no'(False) in the true dataset db.

In [28]:
cat1, cat2 = list(db).count(True), list(db).count(False)
print(cat1, cat2)

48 52


So we can calculate the porpotions of these answer catagories which can be seen as the $Q(D,r)$ in this case.

In [37]:
pro1, pro2 = cat1/len(db), cat2/len(db)
print(pro1, pro2)

0.48 0.52


The sensitivity $\Delta Q$ is $1/len(D)$. The calculation is as follows.

---

**Case 1: Removal**

Let $D$ denote dataset of size $n$, $D'$ denote neighbouring dataset of size $n-1$ and $k$ denote the number of `yes` in $D$.

*Case 1-1: Removing `yes` from $D$*
$$
\Delta Q = \Bigg| Q(D,yes) - Q(D',yes) \Bigg| = \Bigg| \frac{k}{n}-\frac{k-1}{n-1} \Bigg| = \frac{|n-k|}{n(n-1)},
$$
where $k \in [1,n]$. $k$ must be no less than 1 because 1 `yes` is removed from $D$.

> Thus, the greatest $\Delta Q$ in this case is $\frac{1}{n}$ when $k=1$.

*Case 1-2: Removing `no` from $D$*
$$
\Delta Q = \Bigg| \frac{k}{n}-\frac{k}{n-1} \Bigg| = \frac{k}{n(n-1)}.
$$
Similarly, $k \in [0,n-1]$.

> Thus, the greatest $\Delta Q$ in this case is $\frac{1}{n}$ when $k=n-1$.

---

**Case 2: Addition**

Let $D$ denote dataset of size $n$, $D'$ denote neighbouring dataset of size $n+1$ and $k$ denote the number of `yes` in $D$.

*Case 2-1: Adding `yes` to $D$*
$$
\Delta Q = \Bigg| \frac{k}{n} - \frac{k+1}{n+1} \Bigg| = \frac{|k-n|}{n(n+1)},
$$
where $k \in [0,n]$.
> Thus, the greatest $\Delta Q$ in this case is $\frac{1}{n+1}$ when $k=0$.

*Case 2-2: Adding `no` to $D$*
$$
\Delta Q = \Bigg| \frac{k}{n} - \frac{k}{n+1} \Bigg| = \frac{k}{n(n+1)},
$$
where $k \in [0,n]$.
> Thus, the greatest $\Delta Q$ in this case is $\frac{1}{n+1}$ when $k=n$.

---



$\forall n>0, \;\; 1/n \, > \, 1/(n+1) $.

Hence, the sensitivity $\Delta Q = 1/n$.

In [38]:
sensitivity = 1/len(db)
print("sensitivity:", sensitivity)

sensitivity: 0.01


In [39]:
import math
def exponential_mechanism(Q, sensitivity):
    epsilon = 0.5
    pro = math.exp((epsilon*Q)/(2*sensitivity))

    return pro

In [40]:
pro1_after_exp = exponential_mechanism(pro1, sensitivity)
pro2_after_exp = exponential_mechanism(pro2, sensitivity)
print(pro1_after_exp, pro2_after_exp)

162754.79141900392 442413.3920089205


Then we normalise the porpotion to the range (0,1).

In [41]:
total = pro1_after_exp + pro2_after_exp
pro1_after_exp = pro1_after_exp/total
pro2_after_exp = pro2_after_exp/total
print('True porpotion:',cat1/len(db), cat2/len(db) )
print('Noised porpotion:', pro1_after_exp,pro2_after_exp)

True porpotion: 0.48 0.52
Noised porpotion: 0.2689414213699951 0.7310585786300049


# Quiz

### 1. Try different sizes (size=200, 400, ..., 2000) off datasets with the same noise (noise = 0.1) 10 times in local differential privacy, then calculate the average deviation and standard deviation and show your result to the tutor.

### 2. Try different noise (noise=0.1, 0.2, ..., 1) with the same size of dataset (size = 2000) 10 times in local differential privacy, then calculate the average deviation and standard deviation and show your result to the tutor.

### 3. Try different epsilon (0.1, 0.2, ..., 1) in Laplacian mechanism in global differential privacy, and show the result to your tutor.

### 4. Try different epsilon (0.1, 0.2, ..., 1) in exponential mechanism in global differential privacy, and show the result to your tutor.